<a href="https://colab.research.google.com/github/aiden-dm/CSCI-4170/blob/main/CSCI_4170_Homework_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Projects in AI and ML - Homework 6**
##**Aiden Drover-Mattinen**

## Part 1: Transformers

For this part of the assignment, I chose to summarize news articles. I found a dataset containing articles and their human-written summaries from articles from Hindu, Indian Times and Guardian. I have attached a link to the dataset below.

Dataset Link: https://www.kaggle.com/datasets/sunnysai12345/news-summary

### Installs and Imports

In [ ]:
!pip install datasets
!pip install evaluate
!pip install rouge_score
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=c163570ed333a86c344d2d9bfd6c0e0b0dd7fb694d3c3a01ac05cbd7ba627a89
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 9.6 MB/s eta 0:00:00


In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import Dataset
import numpy as np
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import evaluate

### Data Pre-Processing

The first step was to begin pre-processing the data.

In [ ]:
import pandas as pd

# Importing the articles dataset
df = pd.read_csv("/content/drive/MyDrive/CSCI4170_Datasets/news_summary.csv", encoding="latin1")
df.head()

,author,date,headlines,read_more,text,ctext
0,Chhavi Tyagi,"03 Aug 2017,Thursday",Daman & Diu revokes mandatory Rakshabandhan in...,http://www.hindustantimes.com/india-news/raksh...,The Administration of Union Territory Daman an...,The Daman and Diu administration on Wednesday ...
1,Daisy Mowke,"03 Aug 2017,Thursday",Malaika slams user who trolled her for 'divorc...,http://www.hindustantimes.com/bollywood/malaik...,Malaika Arora slammed an Instagram user who tr...,"From her special numbers to TV?appearances, Bo..."
2,Arshiya Chopra,"03 Aug 2017,Thursday",'Virgin' now corrected to 'Unmarried' in IGIMS...,http://www.hindustantimes.com/patna/bihar-igim...,The Indira Gandhi Institute of Medical Science...,The Indira Gandhi Institute of Medical Science...
3,Sumedha Sehra,"03 Aug 2017,Thursday",Aaj aapne pakad liya: LeT man Dujana before be...,http://indiatoday.intoday.in/story/abu-dujana-...,Lashkar-e-Taiba's Kashmir commander Abu Dujana...,Lashkar-e-Taiba's Kashmir commander Abu Dujana...
4,Aarushi Maheshwari,"03 Aug 2017,Thursday",Hotel staff to get training to spot signs of s...,http://indiatoday.intoday.in/story/sex-traffic...,Hotels in Maharashtra will train their staff t...,Hotels in Mumbai and other Indian cities are t...


I extracted only the summaries and articles, and renamed their columns to be more easily understable in my opinion.

In [ ]:
# Extracting the summaries and full articles
data_df = df[["text", "ctext"]].rename(
    columns={"text": "summary", "ctext": "article"}
)

# Display the first few rows
data_df.head()

,summary,article
0,The Administration of Union Territory Daman an...,The Daman and Diu administration on Wednesday ...
1,Malaika Arora slammed an Instagram user who tr...,"From her special numbers to TV?appearances, Bo..."
2,The Indira Gandhi Institute of Medical Science...,The Indira Gandhi Institute of Medical Science...
3,Lashkar-e-Taiba's Kashmir commander Abu Dujana...,Lashkar-e-Taiba's Kashmir commander Abu Dujana...
4,Hotels in Maharashtra will train their staff t...,Hotels in Mumbai and other Indian cities are t...


Next, I checked if there were any empty values in either of the columns, which unfortunatly there were.

In [ ]:
# Ensure that there are no empty rows
print(data_df.isnull().sum())

summary      0
article    118
dtype: int64


As a result, I dropped all columns with empty values.

In [ ]:
# There are empty rows so we need to drop them
data_df = data_df.dropna()

# Count again to make sure they have been removed
print(data_df.isnull().sum())

summary    0
article    0
dtype: int64


Now I define some key variables for my model, including the maximum length of the input (article) and target (summary) sequences. Additionally, I define which pre-trained BART checkpoint to load from.

In [ ]:
# Define variables
max_input = 512
max_target = 128
batch_size = 3
model_checkpoints = "facebook/bart-large-xsum"

# Import the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoints)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.51k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

The next step was to define a function to preprocess the data. It takes a DataFrame containing articles and their summaries, tokenizes both using a pre-loaded tokenizer, and prepares them for model training. It tokenizes the articles into fixed-length input sequences and the summaries into target sequences (labels) with specified maximum lengths. Finally, it returns a dictionary containing the tokenized inputs, attention masks, and labels that can be fed into the model.

In [ ]:
# Create function to pre-process the data
def preprocess_data(data_df):
  # Extract the articles
  inputs = [article for article in data_df['article']]
  # Tokenize the articles
  model_inputs = tokenizer(inputs,  max_length=max_input, padding='max_length', truncation=True)
  # Tokenize the summaries
  with tokenizer.as_target_tokenizer():
    targets = tokenizer(data_df['summary'], max_length=max_target, padding='max_length', truncation=True)

  # Set labels
  model_inputs['labels'] = targets['input_ids']
  #return the tokenized data, input_ids, attention_mask and labels
  return model_inputs

With the preprocessing function defined, I actually had to prepare my data for the model. The following code converts my pandas DataFrame into a Hugging Face Dataset and then splits it into training, validation, and test sets. It first creates a 90/10 train-test split, then further splits the training set into 80% training and 20% validation. Finally, it applies the preprocess_data function to each of these splits—tokenizing the articles and summaries—so that the resulting datasets are ready to be fed into the model for training and evaluation.

In [ ]:
# Convert pandas DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(data_df)

# Split dataset
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)

# Extract train and test sets
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

# Split the original train dataset into train and validation sets (80% train, 20% validation)
train_dataset_split = train_dataset.train_test_split(test_size=0.2, seed=42)

# Extract the new train and validation sets
train_dataset_final = train_dataset_split["train"]
val_dataset = train_dataset_split["test"]

# Apply the preprocessing function to both train and validation datasets
tokenized_train = train_dataset_final.map(preprocess_data, batched=True)
tokenized_val = val_dataset.map(preprocess_data, batched=True)
tokenized_test = test_dataset.map(preprocess_data, batched=True)

Map:   0%|          | 0/3164 [00:00<?, ? examples/s]

Map:   0%|          | 0/792 [00:00<?, ? examples/s]

Map:   0%|          | 0/440 [00:00<?, ? examples/s]

### Training BART

With the data prepared, I was ready to create my model and train it to generate article summaries.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoints)

pytorch_model.bin:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/309 [00:00<?, ?B/s]

For my model to train properly, I need to also define a collator. In this context, it is a function that takes individual examples from my dataset and combines them into a single batch. It handles tasks such as padding variable-length sequences to a consistent length, converting data into tensors, and organizing inputs and labels so they can be processed efficiently by your model during training or evaluation.

In [ ]:
# Define the data collator
collator = DataCollatorForSeq2Seq(tokenizer, model=model)

The follwoing function takes the predicted and reference token IDs from the model's output, decodes them into text, and then calculates ROUGE scores to measure the overlap between the generated summaries and the ground-truth summaries. It provides a quantitative evaluation of the model's summarization performance during training and validation.

In [ ]:
# Load the ROUGE metric using the new evaluate library
rouge = evaluate.load("rouge")

# Define the compute_metrics function
def compute_rouge(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    # Decode the predictions and labels
    decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)

    # Compute ROUGE scores
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels)

    return result

In the following code, I define the hyperparameters for my model, the trainer, and the datasets that the model will train on.

In [ ]:
# Defining the training arguments for BART
args = Seq2SeqTrainingArguments(
    'conversation-summ',
    evaluation_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size= 2,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=2,
    predict_with_generate=True,
    eval_accumulation_steps=3,
    fp16=True, #available only with CUDA
    report_to="none"
)

# Defining the trainer for BART
trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=collator,
    tokenizer=tokenizer,
    compute_metrics=compute_rouge
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-14-ece8a1bf6b8b>:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


This code formally begins the training process, and saves the final weights to checkpoint files because the training process is extremely lengthly.

In [ ]:
# Start the model training process
trainer.train()

# Save the fine-tuned model and tokenizer
model.save_pretrained("/content/drive/MyDrive/CSCI4170_Datasets")
tokenizer.save_pretrained("/content/drive/MyDrive/CSCI4170_Datasets")

Epoch,Training Loss,Validation Loss


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3353: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 62, 'min_length': 11, 'early_stopping': True, 'num_beams': 6, 'no_repeat_ngram_size': 3}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,1.027900,0.856441,0.489464,0.278142,0.374695,0.374804
2,0.648700,0.851370,0.492098,0.280289,0.376128,0.376144


('/content/drive/MyDrive/CSCI4170_Datasets/tokenizer_config.json',
 '/content/drive/MyDrive/CSCI4170_Datasets/special_tokens_map.json',
 '/content/drive/MyDrive/CSCI4170_Datasets/vocab.json',
 '/content/drive/MyDrive/CSCI4170_Datasets/merges.txt',
 '/content/drive/MyDrive/CSCI4170_Datasets/added_tokens.json',
 '/content/drive/MyDrive/CSCI4170_Datasets/tokenizer.json')

This code loads the weight checkpoint files saved after model training. It can be used to avoid having to retrain the model every time from scratch, wasting valuable time.

In [ ]:
# Load the fine-tuned model and tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained("/content/drive/MyDrive/CSCI4170_Datasets")
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/CSCI4170_Datasets")

### Evaluating Using BLEU and ROUGE

The following code computes the model's predictions for the test dataset and then decodes them so they are human-readable.

In [ ]:
import sacrebleu

# Get model predictions
predictions = trainer.predict(tokenized_test).predictions
decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

The assignment requires evaluating my model’s performance using both BLEU and ROUGE metrics. The code below iterates over the first 50 model predictions, printing each generated summary alongside its corresponding human-written reference. It also computes and displays the BLEU and ROUGE scores for every example, providing detailed insight into the quality of the summarizations.

In [ ]:
import sacrebleu

# Load ROUGE metric
rouge = evaluate.load("rouge")

for i in range(len(decoded_preds[:50])):
    pred = decoded_preds[i]
    ref = test_dataset[i]['summary']

    # Compute BLEU score
    bleu = sacrebleu.sentence_bleu(pred, [ref]).score

    # Compute ROUGE scores
    rouge_scores = rouge.compute(predictions=[pred], references=[ref])

    print(f"Prediction: {pred}")
    print(f"Reference: {ref}")
    print(f"BLEU Score: {bleu:.2f}")
    print(f"ROUGE Scores: {rouge_scores}")
    print()


Prediction: Mahatma Gandhi's assassination is being remembered on the 68th anniversary of his death.
Reference: Mahatma Gandhi was assassinated on January 30, 1948, during a prayer meeting at Delhi?s Birla House. He was shot thrice by Nathuram Godse. "If I'm to die by the bullet of a mad man, I must do so smiling. God must be in my heart and on my lips," Gandhi had reportedly said two days before his assassination.
BLEU Score: 0.08
ROUGE Scores: {'rouge1': 0.21621621621621623, 'rouge2': 0.027777777777777776, 'rougeL': 0.16216216216216217, 'rougeLsum': 0.16216216216216217}

Prediction: A World War Two veteran has become the oldest person in the world to skydive.
Reference: Bryson William Verdun Hayes, at the age of 101 years and 38 days, became the oldest person in the world to skydive after jumping from 15,000 feet. Verdun took ten members of his family to skydive along with him. Interestingly, the 101-year-old had been presented with the Legion d'honneur for his heroic actions in Worl

The model's performance varies significantly across different examples. For some inputs—like the predictions about the world’s fastest elevator or the group of Indian sailors—the BLEU and ROUGE scores suggest a decent overlap with the human-written references. In contrast, other examples, such as the summary of Mahatma Gandhi's assassination, yield much lower scores, indicating that the generated summaries diverge in both wording and content.

Overall, the model does a good job of summarizing article content accurately compared to human-written summaries. However, the target length of 128 may be too short to maximize BLEU and ROUGE scores, as many reference summaries exceed 300 characters. This constraint likely prevents the model from capturing additional details present in the human summaries. That said, I find the model's outputs to be more concise, readable, and well-structured than their human counterparts, making them potentially more effective for quick comprehension.

The chosen hyperparameters support stable and efficient training. A low learning rate of 2e-5 ensures gradual updates and helps prevent catastrophic forgetting, while the small batch size (with gradient accumulation effectively processing four examples per update) balances generalization and gradient stability. Training for only two epochs minimizes overfitting risks but may require careful monitoring to avoid underfitting. Additionally, a weight decay of 0.01 helps regularize the model, and mixed precision (fp16=True) speeds up training with minimal impact on accuracy.

Choosing BART for this summarization task has likely contributed to strong performance. BART's architecture combines bidirectional context understanding, similar to BERT, with autoregressive text generation, similar to GPT. This design makes it well-suited for sequence-to-sequence tasks, including abstractive summarization. Studies show that BART achieves state-of-the-art results in NLP, often matching or surpassing models like RoBERTa on benchmarks such as GLUE and SQuAD.

Fine-tuning further enhances BART's effectiveness for text generation, allowing it to produce concise, contextually relevant summaries from complex inputs. Its ability to distill key information while maintaining fluency makes it a strong choice for summarization. Given these strengths, it's likely that BART's architecture played a key role in generating high-quality, coherent, and contextually appropriate summaries.

Sources:
* https://link.springer.com/chapter/10.1007/978-981-19-7346-8_32?utm_source=chatgpt.com
* https://www.digitalocean.com/community/tutorials/bart-model-for-text-summarization-part1?utm_source=chatgpt.com
* https://ciplav.com/abstractive-text-summarization-using-transformers-bart-model/?utm_source=chatgpt.com

## Part 2: Reinforcement Learning

### Task 2

A real-world application of RL that I came across is its use in controlling a helicopter. Specifically, an RL-based approach was applied to the Quanser Aero 2, a one-degree-of-freedom (1-DoF) helicopter testbed. This system is equipped with two motors that control the pitch of the beam by adjusting the voltages applied to them, allowing the RL agent to learn and optimize control strategies for stabilization and movement.

The following is a brief description of the problem's state space, action space, transition model, and rewards:
* **State Space:** At each time step $t$, the state is defined as $(\Theta_t, \omega_t, r_t)$, where $\Theta_t$ is the pitch angle, $\omega_t = \Theta_t - \Theta_{t-1}$ represents angular velocity, and $r_t$ is the target angle. This setup keeps the state representation simple while still capturing the key dynamics of the system, since pitch and angular velocity naturally describe how the helicopter moves.
* **Action Space:** The action space consists of a single continuous action, which controls the voltage applied to the motors. This voltage is constrained by the system's physical limits: $ -24V \leq a \leq 24V $. The motors work in opposition, meaning when one applies a positive voltage, the other applies an equal negative voltage.  
* **Transition Model:** The paper highlights that in many real-world applications, an explicit transition function is not available, which is the case in their helicopter control task. Instead of relying on a predefined transition function, their RL approach interacts with a Simulink virtual environment and a real-world Quanser Aero 2 system. In these environments, state transitions are determined either through numerical integration (in simulation) or by the physical dynamics of the system (in the real-world setup).
* **Rewards:** The reward is defined as the negative absolute difference between the current pitch angle and the target pitch angle: $ R_t = -|\Theta_t - r_t| $. If the system truncates due to exceeding its limits ($|\Theta_t| \geq \pi / 2$), the reward is scaled by the remaining time steps to penalize early termination.  

Source: https://arxiv.org/html/2503.20442v1


### Task 3

Reinforcement Learning (RL) has demonstrated significant potential in healthcare, particularly in optimizing dynamic treatment regimes (DTRs) for patients with chronic illnesses. DTRs involve sequential decision-making to customize treatments based on individual patient responses, aiming to enhance long-term health outcomes.

Managing chronic conditions requires personalized treatment strategies that adapt over time. Traditional methods may not effectively capture the complex-time dependent nature of patient responses to treatments. RL can be used to overcome this problem by learning optimal treatment policies by continously interacting with patient data.

An open source project that has addressed this challenge is called pH-RL, which stands for personalization in e-Health. It is a reinforcement learning personalization architecture that is designed to integrate with mobile applications in e-Health.

The pH-RL framework is designed to personalize e-Health applications by using data from users' smart devices, such as mobile phones and smartwatches. Users provide demographic information and health goals during sign-up, which forms initial clusters for personalization. As users interact with the app, data is continuously gathered, allowing the system to infer contextual information about their behavior without explicit input. States are defined by feature vectors that capture user behavior at a given time, and interventions (actions) are provided through notifications on the devices. Rewards, either direct (e.g., mood ratings) or indirect (e.g., increased activity), are used as feedback to refine the RL model and improve the personalization of interventions to enhance long-term goal adherence.

pH-RL models the personalization task as a Markov Descision Process (MDP), where the system learns optimal intervention strategies through reinforcement learning. The problem is formalized as $M = \langle S, A, T, R \rangle$, where $S$ represents the finite state space capturing user behavior, $A$ is the set of possible interventions (ex. notifications or recommendations), $T$ defines the probabilistic state transition function and $R$ is the reward function that quantifies user engagement or adherence. The goal is to learn an optimal policy $\pi^*$ that selects actions maximizing the expected long-term reward, represented by the Q-function $Q^\pi(s, a)$.

To improve the personalization, pH-RL uses cluster-based policy improvement, and avoids the limitations of one-size-fits-all approaches by grouping users with similar behavior. This is acheived using Dynamic Time Warping (DTW) to measure the similarity of user behavioral traces, followed by clustering algorithms such as K-means or K-medoids. Each cluter learns a separate Q-function and policy, allowing for more personalized decision-making. This approach ensures that interventions are tailored to users with similar engagement patterns, optimizing the effectiveness of health recommendations.

A real-life application of the pH-RL framework was its integration into the MoodBuster platform. In an experiment, the framework was used to deliver personalized motivational feedback messages to users participating in an online course aimed at improving mood and mental health. By tailoring messages based on user mood and engagement, pH-RL sought to enhance adherence to the course. The results showed that the framework significantly improved user retention and engagement with the MoodBuster application. This experiment highlights the potential of RL-driven personalization in mental health support, making it a promising tool for future e-Health applications.

Sources:
* https://capestart.com/resources/blog/reinforcement-learning-in-health-care-why-its-important-and-how-it-can-help/
* https://arxiv.org/pdf/2103.15908

## Part 3: Recommender Systems

For this part of the assignment, I chose to implement the Alternating Least Squares (ALS) and Neural Collaborative Filtering (NLS) recommendation systems.

### Pre-Processing the MovieLens 100k Dataset

The first step was to prepare the 100k Dataset linked in the homework instructions. The first step was to load the content from my Google Drive where I stored the dataset locally.

In [ ]:
import numpy as np
import pandas as pd
import pyspark
from pyspark.sql import SparkSession

In [ ]:
# Initialize Spark session
spark = SparkSession.builder.appName("Homework6").getOrCreate()

# Create SQLContext from SparkSession
sqlContext = pyspark.sql.SQLContext(spark)

# Check Spark version
print(spark.version)

3.5.5


I am storing the data in a PySpark DataFrame because it is the required input format for the library I am using to implement the ALS model. This approach eliminates the need to manually format the data into a user-item matrix, as the library expects a slightly different data structure.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType

# Define the schema for the dataset
schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("item_id", IntegerType(), True),
    StructField("rating", FloatType(), True),
    StructField("timestamp", IntegerType(), True)
])

# Load the dataset
data = spark.read.csv("/content/drive/MyDrive/CSCI4170_Datasets/ml-100k/u.data", sep="\t", schema=schema)

# Drop timestamp since it's not needed for ALS
data = data.drop("timestamp")

# Show data sample
data.show(5)

+-------+-------+------+
|user_id|item_id|rating|
+-------+-------+------+
|    196|    242|   3.0|
|    186|    302|   3.0|
|     22|    377|   1.0|
|    244|     51|   2.0|
|    166|    346|   1.0|
+-------+-------+------+
only showing top 5 rows



To improve model performance, I ensure that there are no missing values and no duplicates in the dataset.

In [ ]:
# Drop any missing values
print(f'Count Before Missing Value Correction: {data.count()}')
data = data.dropna()
print(f'Count After Missing Value Correction: {data.count()}')

# Remove duplicates if necessary
print(f'Count Before Duplicate Removal: {data.count()}')
data = data.dropDuplicates()
print(f'Count After Duplicate Removal: {data.count()}')

# Print schema to ensure it is correct
data.printSchema()

Count Before Missing Value Correction: 100000
Count After Missing Value Correction: 100000
Count Before Duplicate Removal: 100000
Count After Duplicate Removal: 100000
root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- rating: float (nullable = true)



To mitigate the risk of overfitting to underrepresented users, I filter out users who have provided fewer than 5 ratings.

In [ ]:
from pyspark.sql import functions as F

# Count the ratings per user
user_counts = data.groupBy("user_id").count()

# Filter users with at least 5 ratings
active_users = user_counts.filter(F.col("count") >= 5).select("user_id")

# Join the active users with the original data
print(f'Count Before Item Imbalance Correction: {data.count()}')
data = data.join(active_users, on="user_id", how="inner")
print(f'Count After Item Imbalance Correction: {data.count()}')

Count Before Item Imbalance Correction: 100000
Count After Item Imbalance Correction: 100000


I applied a similar approach for underrepresented items, filtering out those with fewer than 5 ratings.

In [ ]:
# Count ratings per item
item_counts = data.groupBy("item_id").count()

# Filter items with at least 5 ratings
popular_items = item_counts.filter(F.col("count") >= 5).select("item_id")

print(f'Count Before Rating Imbalance Correction: {data.count()}')
data = data.join(popular_items, on="item_id", how="inner")
print(f'Count After Rating Imbalance Correction: {data.count()}')

Count Before Rating Imbalance Correction: 100000
Count After Rating Imbalance Correction: 99287


Finally, I split the dataset into training, validation, and test sets with a 60-20-20 distribution.

In [ ]:
from pyspark.sql.window import Window

# Performing train and test split

# Assign a random split to each row, ensuring all users appear in both sets
data = data.withColumn("split", F.rand())

# Define window by user_id to ensure each user appears in both sets
window_spec = Window.partitionBy("user_id").orderBy(F.col("split"))

# Assign row numbers within each user partition
data = data.withColumn("row_number", F.row_number().over(window_spec))

# 60% of data for training, 20% for validation, and 20% for testing
train_data = data.filter(F.col("split") <= 0.6).drop("split", "row_number")
validation_data = data.filter((F.col("split") > 0.6) & (F.col("split") <= 0.8)).drop("split", "row_number")
test_data = data.filter(F.col("split") > 0.8).drop("split", "row_number")

# Print sizes
print(f'Train Size: {train_data.count()}')
print(f'Validation Size: {validation_data.count()}')
print(f'Test Size: {test_data.count()}')

Train Size: 59687
Validation Size: 19876
Test Size: 19724


### Implementing Alternating Least Squares (ALS)

As mentioned earlier, I used the PySpark library to implement the ALS algorithm. I have included a link to the PySpark documentation for this algorithm below as a reference. In the following code, I define the model and train it on the training set.

Source: https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.recommendation.ALS.html

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.sql.types import FloatType
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

In [ ]:
# Define the ALS model
als = ALS(
    userCol = "user_id",
    itemCol = "item_id",
    ratingCol = "rating",
    rank = 10,
    maxIter = 10,
    regParam = 0.1,
    coldStartStrategy = "drop"
)

# Train the model
model = als.fit(train_data)

With the model trained, I generated predictions on the validation set.

In [ ]:
# Make predictions on the validation set
predictions = model.transform(validation_data)
predictions.show(5)

# Print min and max predictions
predictions.select(F.min("prediction"), F.max("prediction")).show()

+-------+-------+------+----------+
|item_id|user_id|rating|prediction|
+-------+-------+------+----------+
|    496|    148|   3.0|  4.031479|
|    496|    897|   5.0| 4.1713977|
|    471|    540|   4.0| 3.3551936|
|    471|    580|   3.0|  3.403893|
|    496|    321|   4.0| 4.0985694|
+-------+-------+------+----------+
only showing top 5 rows

+---------------+---------------+
|min(prediction)|max(prediction)|
+---------------+---------------+
|      0.2828749|      5.6216125|
+---------------+---------------+



Finally, I defined a function to output the two performance metrics I've selected to compare the algorithms, which I will justify in a later section.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

def print_als_performacencf(model, data):
    # Generate predictions
    als_predictions = model.transform(data)

    # Convert predictions to Pandas DataFrame
    als_predictions_pd = als_predictions.select("user_id", "item_id", "prediction").toPandas()
    actual_ratings_pd = data.select("user_id", "item_id", "rating").toPandas()

    # Merge predictions with actual ratings
    als_comparison = als_predictions_pd.merge(actual_ratings_pd, on=["user_id", "item_id"])

    # Compute RMSE and MAE
    als_rmse = np.sqrt(mean_squared_error(als_comparison['rating'], als_comparison['prediction']))
    als_mae = mean_absolute_error(als_comparison['rating'], als_comparison['prediction'])

    print(f"ALS RMSE: {als_rmse}, ALS MAE: {als_mae}")

print_als_performacencf(model, validation_data)

ALS RMSE: 0.8051004183670415, ALS MAE: 0.6303983926773071


### Implementing Neural Collaborative Filtering (NCF)

I implemented the NCF system using the Recommenders library, with a helpful GitHub tutorial as a guide, which I have linked below as a source. Since Recommenders is not installed by default on Google Colab, I also included the necessary installation steps as shown below.

Source: https://github.com/recommenders-team/recommenders/blob/main/examples/00_quick_start/ncf_movielens.ipynb

In [ ]:
!pip install recommenders

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 17.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of notebook to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.3/355.3 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.4/25.4 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.2/264.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 115.4 MB/s eta 0:00:00
   ━━

In [ ]:
from recommenders.models.ncf.ncf_singlenode import NCF
from recommenders.models.ncf.dataset import Dataset as NCFDataset

I made several modifications to my datasets to properly input them into the NCF model. These adjustments included converting them to Pandas dataframes, renaming the columns, and sorting them by userID.

In [ ]:
# Convert the PySpark DataFrame 'data' to a pandas DataFrame
train_data_pd = train_data.toPandas()
validation_data_pd = validation_data.toPandas()
test_data_pd = test_data.toPandas()

# Rename columns to match the expected format for NCFDataset
train_data_pd = train_data_pd.rename(columns={'user_id': 'userID', 'item_id': 'itemID'})
validation_data_pd = validation_data_pd.rename(columns={'user_id': 'userID', 'item_id': 'itemID'})
test_data_pd = test_data_pd.rename(columns={'user_id': 'userID', 'item_id': 'itemID'})

# Sort the dataframes by userID
train_data_pd = train_data_pd.sort_values(by='userID')
validation_data_pd = validation_data_pd.sort_values(by='userID')
test_data_pd = test_data_pd.sort_values(by='userID')

In [ ]:
# Printing the sizes of the dataframes
print(f'Train Size: {train_data_pd.shape}')
print(f'Validation Size: {validation_data_pd.shape}')
print(f'Test Size: {test_data_pd.shape}')

Train Size: (59687, 3)
Validation Size: (19876, 3)
Test Size: (19724, 3)


In [ ]:
# Defining filepaths for data files
train_file = "./train.csv"
validation_file = "./validation.csv"
test_file = "./test.csv"

# Save the pandas DataFrames to CSV files
train_data_pd.to_csv(train_file, index=False)
validation_data_pd.to_csv(validation_file, index=False)
test_data_pd.to_csv(test_file, index=False)

In [ ]:
ncf_data = NCFDataset(train_file=train_file, test_file=validation_file, seed=42)

  0%|          | 0/940 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
  0%|          | 3/940 [00:00<00:33, 28.02it/s]/usr/local/lib/python3.11/dist-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bou

Once the dataset was properly configured for the NCF model, I proceeded to define the model and train it on the training data.

In [77]:
# top k items to recommend
TOP_K = 10

# Model parameters
EPOCHS = 50
BATCH_SIZE = 256
SEED = 42

ncf_model = NCF (
    n_users=ncf_data.n_users,
    n_items=ncf_data.n_items,
    model_type="NeuMF",
    n_factors=16,
    layer_sizes=[32, 16, 8],
    n_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=1e-4,
    verbose=10,
    seed=SEED
)

/usr/local/lib/python3.11/dist-packages/tensorflow/python/keras/engine/base_layer_v1.py:1694: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  warnings.warn('`layer.apply` is deprecated and '


In [78]:
ncf_model.fit(ncf_data)

Similar to the previous system, I defined a function to calculate and display the performance metrics, which I will discuss in a later section.

In [ ]:
def get_ncf_preds(ncf_model, data):
    users, items, preds = [], [], []

    # Get the valid users and items from the trained model
    known_users = set(ncf_model.user2id.keys())
    known_items = set(ncf_model.item2id.keys())

    # Filter data to only include known users and items
    filtered_data = data[(data.userID.isin(known_users)) & (data.itemID.isin(known_items))]

    if filtered_data.empty:
        print("Warning: No valid users or items found in the dataset!")
        return pd.DataFrame(columns=["userID", "itemID", "prediction"])

    # Get unique items in the filtered dataset
    unique_items = list(filtered_data.itemID.unique())

    for user in filtered_data.userID.unique():
        user_batch = [user] * len(unique_items)  # Repeat user for all items
        users.extend(user_batch)
        items.extend(unique_items)
        preds.extend(list(ncf_model.predict(user_batch, unique_items, is_list=True)))

    # Create DataFrame with predictions
    all_predictions = pd.DataFrame({"userID": users, "itemID": items, "prediction": preds})

    # Merge with original data, but keep all rows
    merged = pd.merge(data, all_predictions, on=["userID", "itemID"], how="inner")

    # Return all predictions, keeping ratings if available
    return merged

In [79]:
def print_ncf_performacencf(model, data):
    ncf_predictions = get_ncf_preds(model, data)
    ncf_rmse = np.sqrt(mean_squared_error(ncf_predictions['rating'], ncf_predictions['prediction']))
    ncf_mae = mean_absolute_error(ncf_predictions['rating'], ncf_predictions['prediction'])
    print(f"NCF RMSE: {ncf_rmse}, NCF MAE: {ncf_mae}")

print_ncf_performacencf(ncf_model, validation_data_pd)

NCF RMSE: 3.257377321007435, NCF MAE: 3.0649445056915283


### Performance Comparison

I chose to use RMSE (Root Mean Squared Error) and MAE (Mean Absolute Error) to compare the performance of two recommender system implementations because they provide clear, interpretable insights into prediction accuracy. RMSE penalizes larger errors more heavily, making it useful for cases where minimizing significant mistakes is crucial, while MAE gives a more straightforward measure of average error, unaffected by extreme outliers. By using both metrics, I ensure a balanced evaluation that captures both the typical magnitude of errors (MAE) and the model's sensitivity to large discrepancies (RMSE). This approach helps to assess overall model performance and understand the impact of large errors on recommendation quality.

Source: https://arxiv.org/html/2312.16015v2

In [80]:
print_als_performacencf(model, test_data)
print_ncf_performacencf(ncf_model, test_data_pd)

ALS RMSE: 0.8064648157238785, ALS MAE: 0.6376165747642517
NCF RMSE: 3.2489857191322034, NCF MAE: 3.057892322540283


The ALS model outperforms the NCF model in both RMSE and MAE, indicating better prediction accuracy. For ALS, the RMSE is 0.81 and the MAE is 0.64, indicating that, on average, the model's predictions are relatively close to the true ratings. RMSE gives more weight to larger errors, so a value of 0.81 suggests that most predictions are reasonably accurate with few large deviations. The MAE of 0.64 shows that, on average, the model's predictions are off by about 0.64 rating points.

For NCF, the RMSE is 3.25 and the MAE is 3.06, which are much higher than the ALS metrics. This means the NCF model's predictions are farther from the true ratings, on average. An RMSE of 3.25 and an MAE of 3.06 indicate that NCF is having more difficulty predicting ratings accurately and is likely producing more significant errors compared to ALS.

A big reason behind ALS' improved performance over NCF is that ALS is designed for explicit feedback, like user ratings, while NCF tends to work better for implicit data, like clicks or purchases. ALS directly minimizes squared error, which lines up well with RMSE and MAE, while NCF uses a more complex, non-linear approach that might not be as stable. It's also possible that NCF is struggling with overfitting or that it needs better tuning, but based on these results, ALS seems to be the better choice for this task.

Sources:
* https://arxiv.org/html/2312.16015v2
* https://github.com/recommenders-team/recommenders/blob/main/examples/00_quick_start/ncf_movielens.ipynb
* https://medium.com/@jonahflateman/building-a-recommender-system-in-pyspark-using-als-18e1dd9e38e6

## Statement About Use of Generative AI

Throughout this project, I leveraged generative AI models to help refine the structure and grammar of my writing. I also used them to format some of my code for better readability and consulted them for guidance on which libraries and functions to use before researching and implementing them myself. All content presented is my own, and any external sources referenced for this assignment are properly cited or linked where appropriate. I wasn't sure of the exact citation format for AI usage, so I felt this paragraph would suffice.

I thoroughly enjoyed completing this assignment and deepening my understanding of so many new concepts!